In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Load aggregated forecast
agg_file = 'all_models_forecast_2026.csv'
if not os.path.exists(agg_file):
    print(f"Error: {agg_file} not found.")
    print("Run aggregate_forecasts.py first to generate it.")
    exit()

df = pd.read_csv(agg_file)
df['Date'] = pd.to_datetime(df['Date'])

print("="*70)
print("ENERGY FORECAST MODEL COMPARISON")
print("="*70)
print(f"\nDataset: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Total days: {len(df)}")
print(f"\nColumns loaded:")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
# Extract model columns
solar_cols = [col for col in df.columns if col.startswith('Solar_')]
wind_cols = [col for col in df.columns if col.startswith('Wind_')]

model_names = {
    'Solar_model1_ensemble': 'Ensemble',
    'Solar_model2_sarima': 'SARIMA',
    'Solar_model3_prophet': 'Prophet',
    'Solar_model4_lstm': 'LSTM',
    'Solar_model5_xgboost': 'XGBoost'
}

print(f"\n{'='*70}")
print("SOLAR ENERGY STATISTICS (2026)")
print(f"{'='*70}")

# Calculate statistics for each solar model
solar_stats = {}
for col in sorted(solar_cols):
    model_name = model_names.get(col, col.replace('Solar_', ''))
    data = df[col]
    
    stats = {
        'Mean (kWh/day)': data.mean(),
        'Std (kWh/day)': data.std(),
        'Min (kWh/day)': data.min(),
        'Max (kWh/day)': data.max(),
        'Total (MWh/year)': data.sum() / 1000,
        'CV (%)': (data.std() / data.mean()) * 100  # Coefficient of variation
    }
    solar_stats[model_name] = stats

# Display as table
solar_stats_df = pd.DataFrame(solar_stats).T
print("\n" + solar_stats_df.to_string())

# Calculate pairwise differences
print(f"\n{'='*70}")
print("PAIRWISE FORECAST DIFFERENCES (SOLAR)")
print(f"{'='*70}")
print("\nMean Absolute Difference (MAD) between models (kWh/day):")

model_list = sorted(solar_cols)
mad_matrix = pd.DataFrame(index=[model_names.get(m, m) for m in model_list],
                          columns=[model_names.get(m, m) for m in model_list])

for i, col1 in enumerate(model_list):
    for j, col2 in enumerate(model_list):
        mad = np.mean(np.abs(df[col1] - df[col2]))
        mad_matrix.iloc[i, j] = f"{mad:.2f}"

print("\n" + mad_matrix.to_string())

# Consensus forecast (average of all models)
print(f"\n{'='*70}")
print("CONSENSUS FORECAST (Average of all models)")
print(f"{'='*70}")

df['Solar_Consensus'] = df[solar_cols].mean(axis=1)
consensus_stats = {
    'Mean': df['Solar_Consensus'].mean(),
    'Std': df['Solar_Consensus'].std(),
    'Min': df['Solar_Consensus'].min(),
    'Max': df['Solar_Consensus'].max(),
    'Total (MWh)': df['Solar_Consensus'].sum() / 1000
}

for key, val in consensus_stats.items():
    print(f"{key:20s}: {val:10.2f}")

# Calculate forecast variability (spread across models)
print(f"\n{'='*70}")
print("FORECAST VARIABILITY (Std across models per day)")
print(f"{'='*70}")

df['Solar_ModelStd'] = df[solar_cols].std(axis=1)
print(f"\nMean variability across models: {df['Solar_ModelStd'].mean():.2f} kWh/day")
print(f"Max variability (most disagreement): {df['Solar_ModelStd'].max():.2f} kWh/day on {df.loc[df['Solar_ModelStd'].idxmax(), 'Date'].date()}")
print(f"Min variability (most agreement): {df['Solar_ModelStd'].min():.2f} kWh/day on {df.loc[df['Solar_ModelStd'].idxmin(), 'Date'].date()}")

In [ ]:
if wind_cols:
    print(f"\n{'='*70}")
    print("WIND ENERGY STATISTICS (2026)")
    print(f"{'='*70}")
    
    # Calculate statistics for each wind model
    wind_stats = {}
    for col in sorted(wind_cols):
        model_name = model_names.get(col.replace('Wind_', 'Solar_'), col.replace('Wind_', ''))
        data = df[col]
        
        stats = {
            'Mean (kWh/day)': data.mean(),
            'Std (kWh/day)': data.std(),
            'Min (kWh/day)': data.min(),
            'Max (kWh/day)': data.max(),
            'Total (MWh/year)': data.sum() / 1000,
            'CV (%)': (data.std() / data.mean()) * 100
        }
        wind_stats[model_name] = stats
    
    wind_stats_df = pd.DataFrame(wind_stats).T
    print("\n" + wind_stats_df.to_string())
    
    # Consensus wind forecast
    print(f"\n{'='*70}")
    print("CONSENSUS FORECAST (Wind - Average of all models)")
    print(f"{'='*70}")
    
    df['Wind_Consensus'] = df[wind_cols].mean(axis=1)
    wind_consensus_stats = {
        'Mean': df['Wind_Consensus'].mean(),
        'Std': df['Wind_Consensus'].std(),
        'Min': df['Wind_Consensus'].min(),
        'Max': df['Wind_Consensus'].max(),
        'Total (MWh)': df['Wind_Consensus'].sum() / 1000
    }
    
    for key, val in wind_consensus_stats.items():
        print(f"{key:20s}: {val:10.2f}")

In [ ]:
# Combined solar + wind
print(f"\n{'='*70}")
print("COMBINED ENERGY (Solar + Wind) - 2026 TOTALS")
print(f"{'='*70}")

combined_totals = {}
for col in solar_cols:
    model_name = model_names.get(col, col)
    solar_total = df[col].sum() / 1000  # MWh
    
    if wind_cols:
        wind_col = col.replace('Solar_', 'Wind_')
        if wind_col in df.columns:
            wind_total = df[wind_col].sum() / 1000  # MWh
        else:
            wind_total = 0
    else:
        wind_total = 0
    
    combined_total = solar_total + wind_total
    solar_pct = (solar_total / combined_total * 100) if combined_total > 0 else 0
    wind_pct = (wind_total / combined_total * 100) if combined_total > 0 else 0
    
    combined_totals[model_name] = {
        'Solar (MWh)': solar_total,
        'Wind (MWh)': wind_total,
        'Total (MWh)': combined_total,
        'Solar %': solar_pct,
        'Wind %': wind_pct
    }

combined_df = pd.DataFrame(combined_totals).T
print("\n" + combined_df.to_string())

In [ ]:
# Model Ranking
print(f"\n{'='*70}")
print("MODEL RANKINGS & RECOMMENDATIONS")
print(f"{'='*70}")

print("\n1. FORECAST SPREAD (Lower = More Agreement)")
print(f"   Average disagreement across models: {df['Solar_ModelStd'].mean():.3f} kWh/day")
print(f"   This indicates consensus strength: {'HIGH' if df['Solar_ModelStd'].mean() < 0.5 else 'MODERATE' if df['Solar_ModelStd'].mean() < 1.0 else 'LOW'}")

print("\n2. TOTAL ENERGY PROJECTIONS (2026)")
print("\n   Model Rankings by Total Solar (MWh):")
solar_totals = {model: stats['Total (MWh/year)'] for model, stats in solar_stats.items()}
for i, (model, total) in enumerate(sorted(solar_totals.items(), key=lambda x: x[1], reverse=True), 1):
    print(f"   {i}. {model:15s}: {total:8.1f} MWh")

if wind_cols:
    print("\n   Model Rankings by Total Wind (MWh):")
    wind_totals = {model: stats['Total (MWh/year)'] for model, stats in wind_stats.items()}
    for i, (model, total) in enumerate(sorted(wind_totals.items(), key=lambda x: x[1], reverse=True), 1):
        print(f"   {i}. {model:15s}: {total:8.1f} MWh")

print("\n3. VARIABILITY (CV = Coefficient of Variation)")
print("   Lower CV = More stable/predictable forecast")
print("\n   Solar Model Rankings by Stability (Lower CV):")
solar_cv = {model: stats['CV (%)'] for model, stats in solar_stats.items()}
for i, (model, cv) in enumerate(sorted(solar_cv.items(), key=lambda x: x[1]), 1):
    print(f"   {i}. {model:15s}: {cv:6.2f}% (Std={solar_stats[model]['Std (kWh/day)']:5.2f})")

print("\n4. RECOMMENDATION")
print("\n   For decision-making, use CONSENSUS FORECAST:")
print(f"   - Average of all {len(solar_cols)} models")
print(f"   - Reduces individual model bias")
print(f"   - More robust to outliers")
print(f"   - Forecast saved in 'all_models_forecast_2026.csv' as 'Solar_Consensus'")

print("\n   Individual Model Best Practices:")
print("   - SARIMA: Best for stationary patterns, seasonal trends")
print("   - Prophet: Best for trend changes, holiday effects")
print("   - LSTM: Best for complex non-linear patterns")
print("   - XGBoost: Best with feature interactions, feature importance analysis")
print("   - Ensemble: Simple baseline, good for quick estimates")

In [ ]:
# Save comparison results
print(f"\n{'='*70}")
print("SAVING COMPARISON RESULTS")
print(f"{'='*70}")

# Save full detailed comparison
df.to_csv('model_comparison_full_2026.csv', index=False)
print("✓ Saved: model_comparison_full_2026.csv (all daily forecasts with consensus)")

# Save summary statistics
summary = pd.concat([solar_stats_df])
summary.to_csv('model_comparison_summary.csv')
print("✓ Saved: model_comparison_summary.csv (statistical summary)")

# Save monthly comparison
df['Month'] = df['Date'].dt.to_period('M')
monthly_comparison = df.groupby('Month')[[col for col in df.columns if 'Energy' in col or 'Consensus' in col]].sum()
monthly_comparison.to_csv('model_comparison_monthly_2026.csv')
print("✓ Saved: model_comparison_monthly_2026.csv (monthly totals)")

print(f"\n{'='*70}")
print("✓ MODEL COMPARISON COMPLETE")
print(f"{'='*70}")